## Tools in LLMs

Tools allow an AI model to interact with external systems and perform tasks beyond generating text, such as searching the web, fetching data from a database, calling APIs, or executing code.

Tools are especially important in AI agents, where the model decides when and how to use a tool to accomplish a task.

### Components of a Tool

Tools consist of two main parts:

1. Tool Schema: Defines the tool's name, description, and input arguments, often using a JSON schema. It tells the model what the tool does and what inputs it requires.
2. Tool Implementation: The actual function or coroutine that executes the task when called.

### How Tool Calling Works

1. The developer provides the model with available tool definitions.
2. The model decides whether a tool is needed for the user's request.
3. The model generates a tool call with the required arguments.
4. The application executes the corresponding function.
5. The tool returns its result to the model.
6. The model uses the result to generate a response for the user.

### Example

Suppose an AI assistant has access to a weather tool.

- Tool name: `get_weather`
- Description: Fetches the current weather for a specified city.
- Input: `city = "Gorakhpur"`
- Implementation: Calls a weather API and retrieves the current conditions.
- Output: Weather information that the model can use to answer the user.

Important points:

- The model typically requests a tool call; the application or agent framework executes it.
- A tool call is different from an ordinary text response.
- Tools can connect LLMs to APIs, databases, search engines, and other software.
- Tool calling is a key capability used in agentic AI, but using tools alone does not automatically make an application a fully autonomous agent.

In [5]:
from dotenv import load_dotenv
load_dotenv(r"D:\GenAI\.env", override=True)

True

In [6]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")

response = model.invoke("Why do parrots talk?")
response.content

'Parrots aren’t really “talking” the way humans do—they’re amazing mimics. Their “talking” behavior comes from a combination of anatomy, brain wiring, and social instincts.\n\n### 1. **Vocal anatomy**\n- **Syrinx:** Birds have a voice box called the syrinx located at the base of the trachea. In parrots it’s especially well‑developed, giving them fine control over pitch, tone, and rhythm.\n- **Tongue and beak:** Parrots can shape sounds with a highly mobile tongue and a flexible beak, allowing them to reproduce a wide range of human phonemes.\n\n### 2. **Brain and learning**\n- **Song‑learning circuitry:** Like songbirds, parrots possess brain regions (the “song system”) that are specialized for hearing, memorizing, and reproducing complex sounds.\n- **High cognitive ability:** Parrots have relatively large forebrains for birds, supporting problem solving, memory, and social cognition—all useful for learning and using vocalizations.\n\n### 3. **Social motivation**\n- **Flock communicati

In [7]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the live weather of the given location."""
    return f"The current weather in {location} is sunny."

model_with_tools = model.bind_tools([get_weather])



In [8]:
response = model_with_tools.invoke(["What's the weather like in London?"])
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in London. We need to call get_weather function.', 'tool_calls': [{'id': 'fc_2fe3c5d7-0505-4ec1-bf76-50f99bddd367', 'function': {'arguments': '{"location":"London"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 129, 'total_tokens': 172, 'completion_time': 0.090771381, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.006258282, 'prompt_tokens_details': None, 'queue_time': 0.210443955, 'total_time': 0.097029663}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1840c-ab72-7788-b133-73efbf91824a-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'London'}, 'id': 'fc_2fe3c5d7-0505-4ec1-bf76-50f99bddd367', 'type': 'tool_call'}], invalid_tool_calls=[], 

In [9]:
for tool_call in response.tool_calls:
    # View the tool calls made by the modell
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'London'}


### Tool Execution Loops

In [10]:
# Step1 : Model generates tool calls
messages = [{"role":"user","content":"What is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston right now is sunny.


In [11]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the weather in Boston?" We need to fetch live weather. Use function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_7f18f47d-c2f9-4391-84c9-f2da86aee177', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 129, 'total_tokens': 185, 'completion_time': 0.117032464, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.005205851, 'prompt_tokens_details': None, 'queue_time': 0.209915129, 'total_time': 0.122238315}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5a93aea882', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1840c-ad36-722e-8587-a7bfbbd49928-0', tool_calls=[{'name': 'get_weather', 'args